In [2]:
# =============================================================================
# NOTEBOOK: load_dimension (Versión Refactorizada)
#
# DESCRIPCIÓN:
#   Notebook genérico para cargar y mantener una tabla de dimensión en la capa Gold.
#   Asegura un esquema estandarizado con columnas de historial y maneja el ciclo
#   de vida completo de los datos (inserciones, actualizaciones SCD1/SCD2 y bajas lógicas).
#
# PARÁMETROS:
#   - task_id_param (integer): El id de la tarea a ejecutar, definido en la tabla de control.
# =============================================================================

# --- Importaciones ---
from pyspark.sql.functions import col, lit, md5, concat_ws, row_number, max as spark_max, expr, when, upper, regexp_replace, trim
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
from datetime import datetime
import time
import random

# --- Parámetros ---
task_id_param = 34 # Reemplazar con el parámetro del pipeline: notebookutils.widgets.get("task_id")

# --- Configuración Global ---
control_table_full_name = "lh_control.dbo.silver_to_gold_control" 
current_utc_timestamp_str = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

StatementMeta(, 75573844-ba48-4682-a4cc-de79d7403f66, 4, Finished, Available, Finished)

In [ ]:
# --- Listas Maestras para Inventarios (DOMAIN_IC) ---
IC_GRPS_1 = ['LAN','LNC','LND','LNF','LNI','PAN','PLD','PLR','PR6','PR660','PR8','MTN','ARMS']
IC_TYPE_HOJ = ['HOJ','HDE','HGR','HDC','HDF']
IC_GRPS_HOJ = ['LAN','LNC','LND','LNF','LNI','PAN','PLA','PLD','PLR','PR6','PR660','PR8','PL1','PL3','PL4','PL5','PL6','PL7','PL8','PLG','PGR','ARMS']
IC_GRPS_FCO = ['ANG','CUA','RED','SOL','SOLM','SOS']
IC_GRPS_HSS = ['HSI','HSN']
IC_GRPS_TUB = ['PTR','TNE','TNM','TUB','ARMS']
IC_GRPS_OEU_FGO = ['CWE','MISC','PLAT','PUER','RAIL','RAILD','RAILS']
IC_GRPS_OEU_RAW = ['MISC','RAIL','RAILS']
IC_GRPS_OEU_WIP = ['CWE','MISC','RAIL','RAILD','RAILS','PUER']
IC_GRPS_TGM = ['CHAS','TRAYS','LOBA','ROLL','JEWEL','ARMS']
IC_GRPS_PL_REF = ['PL1','PL3','PL4','PL5','PL6','PL7','PL8','PLG','PGR']

def apply_domain_rules(df, rule_name):
    """
    Aplica reglas de re-cálculo de company_code (dominio) basadas en metadatos.
    Solo afecta si company_code original es 'SCO'.
    """
    if not rule_name or rule_name == 'NONE':
        return df

    print(f"--- Applying Business Rule: {rule_name} ---")

    # Definimos columnas alias para escribir menos
    # AJUSTA ESTOS NOMBRES A TUS COLUMNAS REALES EN EL DATAFRAME
    c_company = col("company_code") 
    
    # -------------------------------------------------------
    # CASO 1: REGLA DE CLIENTES (DOMAIN_AR) - Requiere JOIN
    # -------------------------------------------------------
    if rule_name == 'DOMAIN_AR':
        # Nota: Asume que tienes acceso
        try:
            # 1. Traemos la tabla de lookup (solo lo necesario)
            lookup_df = spark.sql("""
                SELECT DISTINCT customer_id, region, company_code 
                FROM lh_silver_erp.md.customers 
                WHERE company_code = 'SCO'
            """)
            
            # 2. Preparamos keys para join (limpieza)
            # Asumimos que la columna en tu df origen se llama 'customer_id'
            df_join = df.withColumn("join_key", upper(regexp_replace(col("customer_id"), "&", "")))
            lookup_join = lookup_df.withColumn("lookup_key", upper(regexp_replace(col("customer_id"), "&", "")))
            
            # 3. Join Left
            df = df_join.join(lookup_join, df_join.join_key == lookup_join.lookup_key, "left")
            
            # 4. Aplicar lógica: Solo si es SCO y la region es 61 -> SCM
            # Si no hace match en lookup (cm_region is null), se queda como estaba.
            df = df.withColumn("company_code", 
                when((c_company == 'SCO') & (col("region") == '61'), lit("SCM"))
                .otherwise(c_company)
            )
            
            # 5. Limpieza
            df = df.drop("join_key", "lookup_key", "customer_id", "region")
            
        except Exception as e:
            print(f"WARNING: Failed to apply DOMAIN_AR. Error: {e}")

    # -------------------------------------------------------
    # LÓGICA SIN JOINS (DOMAIN_IC, DOMAIN_GL)
    # -------------------------------------------------------
    else:
        # Aquí construimos una condición booleana compleja llamada "is_scm_condition"
        is_scm_condition = lit(False) # Por defecto falso
        
        if rule_name == 'DOMAIN_GL':
            # Si la entidad es SCM, entonces es SCM
            is_scm_condition = (col("entity") == 'SCM')
            
        elif rule_name == 'DOMAIN_IC':
            # Alias de columnas de inventario
            pl = col("prod_line")
            typ = col("item_type")
            grp = col("item_group")
            
            # Traducimos la lógica gigante a una sola condición booleana
            is_scm_condition = (
                ((typ == 'ROL') & (grp.isin(IC_GRPS_1))) |
                ((typ.isin(IC_TYPE_HOJ)) & (grp.isin(IC_GRPS_HOJ))) |
                ((typ == 'FCO') & (grp.isin(IC_GRPS_FCO))) |
                ((typ == 'HSS') & (grp.isin(IC_GRPS_HSS))) |
                ((typ == 'TUB') & (grp.isin(IC_GRPS_TUB))) |
                ((pl == 'OEU') & (typ == 'FGO') & (grp.isin(IC_GRPS_OEU_FGO))) |
                ((pl == 'OEU') & (typ == 'RAW') & (grp.isin(IC_GRPS_OEU_RAW))) |
                ((pl == 'OEU') & (typ == 'WIP') & (grp.isin(IC_GRPS_OEU_WIP))) |
                ((pl == 'STU') & (typ == 'FGO') & (grp == 'TUB')) |
                ((pl == 'SEL') & (typ == 'FGO') & (grp == 'ESL')) |
                ((pl == 'STU') & (typ == 'FGO') & (grp == 'RAM')) |
                ((pl == 'TEC') & (typ.isin(['FGO','WIP'])) & (grp == 'RAM')) |
                ((pl == 'CNH') & (typ.isin(['FGO','WIP','RAW'])) & (grp == 'MISC')) |
                ((pl == 'TGM') & (typ.isin(['FGO','RAW','WIP'])) & (grp.isin(IC_GRPS_TGM))) |
                (pl == 'REF') |
                (grp.isin(IC_GRPS_PL_REF)) |
                ((typ == 'FGO') & (grp == 'LAS')) |
                ((typ == 'HER') & (grp == 'MDE')) |
                ((pl == 'MAQ') & (typ.isin(['FGO','RAW','WIP']))) |
                (pl.isin(['CME','SME','NSE','JIN'])) |
                ((pl == 'COM') & (typ == 'RAW') & (grp == 'TUB')) |
                (grp.isin(['DOLLY','DUMPBOX']))
            )

        # -------------------------------------------------------
        # APLICACIÓN FINAL DE LA CONDICIÓN
        # -------------------------------------------------------
        # La lógica es:
        # 1. Si el código actual NO es 'SCO' -> Déjalo como está (SCM, BRA, USA, etc.)
        # 2. Si el código ES 'SCO' Y cumple la "is_scm_condition" -> Cámbialo a 'SCM'
        # 3. Si es 'SCO' y NO cumple -> Se queda 'SCO' (el .otherwise maneja esto al mantener original)
        
        df = df.withColumn("company_code", 
            when((c_company == 'SCO') & (is_scm_condition), lit("SCM"))
            .otherwise(c_company)
        )

    return df

In [3]:
# --- Control de concurrencia ---
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_delta_operation_with_retry(fn, operation_name: str = "operación Delta") -> None:
    """Ejecuta una operación Delta (MERGE, append, etc.) con reintentos ante ConcurrentAppendException."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fn()
            if attempt > 1:
                print(f"   {operation_name} completada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada en {operation_name} (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

# --- Función de Logging ---
def update_task_status(status, message):
    """Actualiza la tabla de control con el estado final de la ejecución (con reintentos ante concurrencia)."""
    safe_message = message.replace("'", "''")
    update_query = f"""
        UPDATE {control_table_full_name}
        SET
            last_run_status = '{status}',
            last_message = '{safe_message}',
            last_run_at = CAST('{current_utc_timestamp_str}' AS TIMESTAMP)
        WHERE
            task_id = {task_id_param}
    """

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            print(f"Log updated: Status='{status}', Message='{message}'" + (f" (intento {attempt})" if attempt > 1 else ""))
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada al actualizar control (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"FATAL: Could not update control table log. Reason: {e}")
                raise
    if last_error is not None:
        raise last_error

# --- Bloque Principal de Ejecución ---
source_df = None
target_df = None
try:
    # 1. LEER METADATOS DE LA TAREA
    print(f"--- Starting Dimension Task ID: {task_id_param} ---")
    config_df = spark.sql(f"SELECT * FROM {control_table_full_name} WHERE task_id = {task_id_param} AND is_enabled = true")
    
    if config_df.isEmpty():
        raise ValueError(f"Task ID '{task_id_param}' not found or is disabled.")
        
    config = config_df.first()
    load_type = config["load_type"]
    source_lakehouse = config["source_lakehouse"]
    source_schema = config["source_schema"]
    source_table = config["source_table"]
    target_lakehouse = config["target_lakehouse"]
    target_schema = config["target_schema"]
    target_table = config["target_table"]
    business_keys_str = config["business_keys"]
    surrogate_key = config["surrogate_key"]
    source_filter = config["source_filter"]
    # Se revierte al uso de 'watermark_column' para la deduplicación.
    watermark_column = config["watermark_column"]

    source_object_full_name = f"{source_lakehouse}.{source_schema}.{source_table}"
    target_table_full_name = f"{target_lakehouse}.{target_schema}.{target_table}"
    business_keys_list = [key.strip() for key in business_keys_str.split(',')]

    # 2. (AJUSTE) MARCAR SI HAY QUE CREAR LA TABLA; LA CREACIÓN REAL SE HACE TRAS DEDUP
    history_cols = {"is_current": "BOOLEAN", "start_date": "TIMESTAMP", "end_date": "TIMESTAMP"}
    needs_create = False
    if not spark.catalog.tableExists(target_table_full_name):
        print(f"Target table '{target_table_full_name}' does not exist. Will create with full schema after source dedup...")
        needs_create = True

    # 3. LEER DATOS DE ORIGEN Y DEDUPLICAR
    source_query = f"SELECT * FROM {source_object_full_name}"
    if source_filter:
        source_query += f" WHERE {source_filter}"

    unclean_source_df = spark.sql(source_query)
    print("Deduplicating source data to ensure business key uniqueness...")
    # Se revierte al uso de 'watermark_column' para ordenar la deduplicación.
    window_spec_dedup = Window.partitionBy(*business_keys_list).orderBy(col(watermark_column).desc())
    source_df = unclean_source_df.withColumn("row_num", row_number().over(window_spec_dedup)) \
                                 .filter(col("row_num") == 1).drop("row_num")

    # === NUEVA SECCIÓN DE TRANSFORMACIÓN ===
    print("Checking for business rule transformations...")
    # Pasamos el nombre de la tabla destino para saber qué regla aplicar
    #source_df = apply_business_rules(source_df, target_table)
    # =======================================
    
    source_df.cache()
    print(f"Source data ready. Original records: {unclean_source_df.count()}, After deduplication: {source_df.count()}")

    # 2b. (AJUSTE) GESTIONAR ESQUEMA DE TABLA DE DESTINO AHORA QUE TENEMOS source_df
    if needs_create:
        print("Creating target table with full schema from source_df + SCD columns...")
        empty_target_df = (
            source_df
            .withColumn(surrogate_key, lit(None).cast("int"))
            .withColumn("is_current", lit(True).cast("boolean"))
            .withColumn("start_date", lit("1900-01-01").cast("timestamp"))
            .withColumn("end_date", lit(None).cast("timestamp"))
            .limit(0)
            .select([surrogate_key] + source_df.columns + ["is_current", "start_date", "end_date"])
        )
        (empty_target_df.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(target_table_full_name))
        print("Table created with full schema.")

    # Si la tabla existe (o ya la acabamos de crear), verificar y agregar columnas faltantes del source + history
    target_columns_schema = spark.table(target_table_full_name).schema
    target_cols_lower = {f.name.lower() for f in target_columns_schema.fields}

    # Columnas del source que falten
    source_schema = source_df.schema
    missing_source_cols = []
    for f in source_schema.fields:
        if f.name.lower() not in target_cols_lower:
            # Usamos el tipo en SQL de Spark (simpleString) que Delta entiende
            missing_source_cols.append(f"`{f.name}` {f.dataType.simpleString()}")

    # Columnas de historia por si faltan
    missing_hist_cols = []
    for c, t in history_cols.items():
        if c.lower() not in target_cols_lower:
            missing_hist_cols.append(f"`{c}` {t}")

    cols_to_add_stmt = ", ".join(missing_source_cols + missing_hist_cols)
    if cols_to_add_stmt:
        alter_statement = f"ALTER TABLE {target_table_full_name} ADD COLUMNS ({cols_to_add_stmt})"
        spark.sql(alter_statement)
        print(f"Added missing columns: {cols_to_add_stmt}")
        # Inicializa is_current/start_date si aplicara (no es obligatorio si la tabla está vacía)
        spark.sql(f"UPDATE {target_table_full_name} SET is_current = true, start_date = '1900-01-01' WHERE is_current IS NULL")

    # 4. LEER DATOS DE DESTINO Y OBTENER MAX KEY
    target_df = spark.table(target_table_full_name)
    target_df.cache()
    
    max_key_row = target_df.agg(spark_max(col(surrogate_key))).first()
    max_key = max_key_row[0] if max_key_row[0] is not None else 0
    print(f"Current max {surrogate_key} in '{target_table_full_name}' is: {max_key}")

    # 5. GESTIONAR BAJAS LÓGICAS (SOFT DELETES) - Lógica Unificada
    print("Identifying records deleted from source to deactivate in Gold...")
    target_current_df = target_df.filter(col("is_current") == True)
    records_to_deactivate = target_current_df.join(source_df, business_keys_list, "left_anti")

    if not records_to_deactivate.isEmpty():
        count_deletes = records_to_deactivate.count()
        print(f"Found {count_deletes} records to deactivate.")
        keys_to_deactivate = records_to_deactivate.select(surrogate_key)

        def _do_deactivation_merge():
            t = DeltaTable.forName(spark, target_table_full_name)
            (t.alias("target").merge(keys_to_deactivate.alias("source"), f"target.{surrogate_key} = source.{surrogate_key}")
                .whenMatchedUpdate(set={"is_current": lit(False), "end_date": expr("current_timestamp()")})
                .execute())
        run_delta_operation_with_retry(_do_deactivation_merge, "MERGE de desactivación")
        print(f"Deactivation complete for {count_deletes} records.")
    else:
        print("No records found for deactivation.")

    # 6. APLICAR LÓGICA SCD
    update_cols = [c for c in source_df.columns if c not in business_keys_list]
    source_df_hash = source_df.withColumn("row_hash", md5(concat_ws("||", *update_cols)))
    target_current_df_hash = target_current_df.withColumn("row_hash", md5(concat_ws("||", *update_cols)))
    join_condition_expr = " AND ".join([f"s.`{key}` = t.`{key}`" for key in business_keys_list])
    
    # Se recarga el dataframe de destino actual después de las posibles bajas lógicas
    target_current_df = spark.table(target_table_full_name).filter(col("is_current") == True)

    # 6a. Lógica para SCD Tipo 1
    if load_type == 'SCD1':
        updates_df = source_df_hash.alias("s").join(target_current_df_hash.alias("t"), expr(join_condition_expr)) \
                                   .where(col("s.row_hash") != col("t.row_hash")).select("s.*")
        if not updates_df.isEmpty():
            # Creamos un diccionario para la actualización explícita
            update_dictionary = { col_name: f"source.`{col_name}`" for col_name in update_cols }
            merge_cond = " AND ".join([f"target.`{key}` = source.`{key}`" for key in business_keys_list])

            def _do_scd1_update_merge():
                t = DeltaTable.forName(spark, target_table_full_name)
                (t.alias("target").merge(updates_df.alias("source"), merge_cond).whenMatchedUpdate(set=update_dictionary).execute())
            run_delta_operation_with_retry(_do_scd1_update_merge, "MERGE SCD1 (actualizaciones)")
            print(f"SCD1: Updated {updates_df.count()} records.")
        else:
            print("No records to update.")

        new_records_df = source_df.join(target_current_df, business_keys_list, "left_anti")
        if not new_records_df.isEmpty():
            window_spec_insert = Window.orderBy(*business_keys_list)
            insert_df = new_records_df.withColumn("row_num", row_number().over(window_spec_insert)) \
                                      .withColumn(surrogate_key, col("row_num") + max_key) \
                                      .withColumn("is_current", lit(True)) \
                                      .withColumn("start_date", expr("current_timestamp()")) \
                                      .withColumn("end_date", lit(None).cast("timestamp")) \
                                      .drop("row_num")

            def _do_scd1_insert():
                insert_df.write.format("delta").mode("append").saveAsTable(target_table_full_name)
            run_delta_operation_with_retry(_do_scd1_insert, "Inserción SCD1")
            print(f"SCD1: Inserted {new_records_df.count()} new records.")
        else:
            print("No new records to insert.")
            
    # 6b. Lógica para SCD Tipo 2
    elif load_type == 'SCD2':
        changed_df = source_df_hash.alias("s").join(target_current_df_hash.alias("t"), expr(join_condition_expr)) \
                                   .where(col("s.row_hash") != col("t.row_hash"))
        
        new_df = source_df.join(target_current_df, business_keys_list, "left_anti")

        if not changed_df.isEmpty():
            keys_to_expire = changed_df.select(col(f"t.{surrogate_key}").alias(surrogate_key))

            def _do_scd2_expire_merge():
                t = DeltaTable.forName(spark, target_table_full_name)
                (t.alias("target").merge(keys_to_expire.alias("source"), f"target.{surrogate_key} = source.{surrogate_key}")
                    .whenMatchedUpdate(set={"is_current": lit(False), "end_date": expr("current_timestamp()")})
                    .execute())
            run_delta_operation_with_retry(_do_scd2_expire_merge, "MERGE SCD2 (expiración)")
            print(f"SCD2: Expired {changed_df.count()} records.")
        else:
            print("No records to Expired.")
        
        staged_inserts = changed_df.select("s.*").unionByName(new_df)
        if not staged_inserts.isEmpty():
            window_spec_insert = Window.orderBy(*business_keys_list)
            insert_df = staged_inserts.drop("row_hash") \
                .withColumn("row_num", row_number().over(window_spec_insert)) \
                .withColumn(surrogate_key, col("row_num") + max_key) \
                .withColumn("is_current", lit(True)) \
                .withColumn("start_date", expr("current_timestamp()")) \
                .withColumn("end_date", lit(None).cast("timestamp")) \
                .drop("row_num")

            def _do_scd2_insert():
                insert_df.write.format("delta").mode("append").saveAsTable(target_table_full_name)
            run_delta_operation_with_retry(_do_scd2_insert, "Inserción SCD2")
            print(f"SCD2: Inserted {staged_inserts.count()} new/updated versions.")
        else:
            print("No new records to insert.")

    else:
        raise ValueError(f"Load Type '{load_type}' is not supported for dimensions.")

    update_task_status('Success', f'Task completed successfully. Load type: {load_type}.')

except Exception as e:
    error_message = str(e).replace('\n', ' ').replace('\r', '')
    print(f"ERROR: An exception occurred: {error_message}")
    update_task_status('Failed', error_message)
finally:
    if source_df: source_df.unpersist()
    if target_df: target_df.unpersist()
    print("--- Proceso Finalizado ---")

StatementMeta(, 75573844-ba48-4682-a4cc-de79d7403f66, 5, Finished, Available, Finished)

--- Starting Dimension Task ID: 34 ---
Target table 'lh_gold_master_data.dimentions.dim_periods' does not exist. Will create with full schema after source dedup...
Deduplicating source data to ensure business key uniqueness...
Source data ready. Original records: 612, After deduplication: 612
Creating target table with full schema from source_df + SCD columns...
Table created with full schema.
Current max period_key in 'lh_gold_master_data.dimentions.dim_periods' is: 0
Identifying records deleted from source to deactivate in Gold...
No records found for deactivation.
No records to update.
SCD1: Inserted 0 new records.
Log updated: Status='Success', Message='Task completed successfully. Load type: SCD1.'
--- Proceso Finalizado ---
